# Validación: cómo se parte el tiempo y por qué

**Finalidad**: ver los tramos, los pliegues *walk-forward*, la purga y la parada temprana, y qué otras formas de validar existen.
**Flujo**: Tres tramos → Pliegues → Purga → Parada → Resultado por pliegue → Alternativas
**Requiere**: `make prepare` y los runs de `runs/`. No entrena.
**Decisiones que ilustra**: apartado 5 de `docs/DECISIONES.md`.

In [1]:
import json

import pandas as pd

from oil_rnn import explore as ex
from oil_rnn import explore_plots as plots
from oil_rnn.config import load_config
from oil_rnn.data import load_train_val, training_bounds
from oil_rnn.paths import SPLITS_MANIFEST

cfg = load_config()
time_col, target = cfg["data"]["time_column"], cfg["data"]["target"]
manifest = json.loads(SPLITS_MANIFEST.read_text(encoding="utf-8"))


dev = load_train_val()
completo = ex.pipeline_stages(cfg)["inicio"]
pd.DataFrame(manifest["ranges"], index=["desde", "hasta"]).T.assign(
    filas={k: v["rows"] for k, v in manifest["splits"].items()}
)

,desde,hasta,filas
train,2000-08-30 00:00:00,2019-04-22 00:00:00,4454
val,2019-04-23 00:00:00,2023-01-24 00:00:00,955
test,2023-01-25 00:00:00,2026-11-03 00:00:00,955


## 1. Tres tramos consecutivos

Nunca se mezclan fechas. Train y val forman el tramo de desarrollo, donde se decide todo. Test se usa una vez, al final.

In [2]:
plots.splits_timeline(completo, time_col, target, manifest["ranges"])

<Figure size 1000x320 with 1 Axes>

## 2. Pliegues *walk-forward* de ventana creciente

Un solo tramo de validación puede salir bien o mal por el régimen que le toque. `make cv` repite el experimento 4 veces: cada pliegue ajusta con todo lo anterior y predice los 2 años siguientes.

- **Azul**: ajusta los pesos.
- **Naranja**: no ajusta pesos; decide en qué época parar.
- **Verde**: se predice y se mide.
- **Blanco entre tramos**: purga, filas que no se usan.

La última fila es el modelo final: mismo procedimiento, con test como bloque siguiente.

In [3]:
h = 5
tabla = ex.fold_table(cfg, dev, horizon=h)
display(tabla)
plots.folds_timeline(tabla, h, manifest["ranges"]["test"])

,ajuste_desde,ajuste_hasta,parada_desde,parada_hasta,eval_desde,eval_hasta,filas_ajuste,filas_parada,filas_eval,filas_purga
pliegue,,,,,,,,,,
1,2000-08-30,2014-02-10,2014-02-18,2015-02-16,2015-02-23,2017-02-10,3133,252,504,8
2,2000-08-30,2016-02-04,2016-02-11,2017-02-06,2017-02-13,2019-02-05,3637,252,504,8
3,2000-08-30,2018-01-24,2018-01-31,2019-01-30,2019-02-06,2021-01-26,4141,252,504,8
4,2000-08-30,2020-01-17,2020-01-24,2021-01-20,2021-01-27,2023-01-24,4645,252,504,8
final,2000-08-30,2022-01-12,2022-01-19,2023-01-18,NaT,NaT,5149,252,0,8


<Figure size 1000x415 with 1 Axes>

In [4]:
h = 252
tabla_252 = ex.fold_table(cfg, dev, horizon=h)
display(tabla_252[["ajuste_hasta", "parada_desde", "parada_hasta", "eval_desde", "filas_purga"]])
plots.folds_timeline(tabla_252, h, manifest["ranges"]["test"])

,ajuste_hasta,parada_desde,parada_hasta,eval_desde,filas_purga
pliegue,,,,,
1,2012-02-22,2013-02-26,2014-02-24,2015-02-23,502
2,2014-02-25,2015-02-24,2016-02-18,2017-02-13,502
3,2016-02-19,2017-02-14,2018-02-06,2019-02-06,502
4,2018-02-07,2019-02-07,2020-01-30,2021-01-27,502
final,2020-01-31,2021-01-28,2022-01-25,NaT,502


<Figure size 1000x415 with 1 Axes>

A 252 filas la purga ocupa casi 2 años. El modelo final de ese horizonte ajusta sus pesos solo hasta enero de 2020: no aprende de la pandemia ni de 2022 (decisión 5.4).

## 3. Purga: qué pasaría sin ella

El objetivo de la fila `t` es el precio de `t+h`. Si se entrena hasta la víspera del bloque, los últimos objetivos de entrenamiento son precios de dentro del bloque que se va a predecir.

In [5]:
inicio = 4000  # una frontera cualquiera, en filas
filas = []
for h in (5, 21, 63, 252):
    fit_end, stop_start, stop_end = training_bounds(inicio, h, int(cfg["cv"]["holdout"]))
    filas.append(
        {
            "horizonte": h,
            "primer pronóstico se hace en la fila": inicio - h,
            "último objetivo usado para parar": stop_end - 1,
            "objetivos del futuro sin purga": h - 1,
            "filas sin usar (dos purgas)": (stop_start - fit_end) + (inicio - stop_end),
        }
    )
pd.DataFrame(filas).set_index("horizonte")

,primer pronóstico se hace en la fila,último objetivo usado para parar,objetivos del futuro sin purga,filas sin usar (dos purgas)
horizonte,,,,
5,3995,3995,4,8
21,3979,3979,20,40
63,3937,3937,62,124
252,3748,3748,251,502


«Objetivos del futuro sin purga»: precios posteriores al primer pronóstico que la red habría visto como respuesta correcta. Con purga son 0 (lo comprueba `test_purga_sin_targets_posteriores_al_origen`).

## 4. Parada temprana: cuándo deja de aprender la red

Azul: pérdida en el tramo de ajuste. Naranja: pérdida en el tramo de parada. El punto es la época que se guarda. Si la naranja sube desde el principio, la red memoriza train sin generalizar.

In [6]:
runs = ex.runs_table()
finales = runs[runs["prefix"] == "final"].sort_values("horizon")
rejilla = runs[runs["prefix"] == "g1"]
elegidos = {
    int(row["horizon"]): ex.same_config(rejilla, row).iloc[0] for _, row in finales.iterrows()
}

for h in (5, 63):
    e = elegidos[h]
    red = f"{e['cell'].upper()} · oculto {e['hidden_size']} · capas {e['num_layers']}"
    print(f"h = {h}: {red} · lr {e['learning_rate']:g}")
    display(plots.loss_curves(ex.loss_history(e["run_id"])))

h = 5: GRU · oculto 256 · capas 1 · lr 0.001


<Figure size 1340x300 with 4 Axes>

h = 63: GRU · oculto 128 · capas 2 · lr 0.0001


<Figure size 1340x300 with 4 Axes>

In [7]:
pd.DataFrame(
    {h: ex.fold_metrics(e["run_id"])["best_epoch"].astype(int) for h, e in elegidos.items()}
).rename_axis("pliegue").rename(columns=lambda h: f"mejor época, h = {h}")

,"mejor época, h = 5","mejor época, h = 21","mejor época, h = 63","mejor época, h = 252"
pliegue,,,,
1,1,11,2,2
2,2,27,5,3
3,1,14,3,12
4,2,12,18,1


En varios pliegues la mejor época es la 1 o la 2: la red apenas mejora sobre su punto de partida (decisión 6.4).

## 5. Resultado por pliegue: la media esconde la dispersión

Mejora de RMSE frente a la persistencia en cada bloque, para la configuración elegida en cada horizonte. Por encima de 0, la red es mejor.

In [8]:
for h, e in elegidos.items():
    por_pliegue = ex.fold_metrics(e["run_id"])
    display(
        plots.bars_by_group(
            por_pliegue,
            "skill_vs_baseline",
            f"h = {h} · {e['cell'].upper()} · mejora de RMSE por pliegue (%)",
            xlabel="pliegue",
        )
    )

<Figure size 600x300 with 1 Axes>

<Figure size 600x300 with 1 Axes>

<Figure size 600x300 with 1 Axes>

<Figure size 600x300 with 1 Axes>

A 63 filas la mejora media sale casi entera del pliegue 4 (2021–2023). Esa mejora no se repitió en test (decisión 7.4).

## 6. Otras formas de validar

| Esquema | Qué hace | Problema |
|---|---|---|
| Un solo tramo | Train y un bloque | Depende de un régimen |
| *k-fold* barajado | Mezcla fechas | Entrena con el futuro |
| Ventana deslizante | Ajusta con los últimos N años | Olvida regímenes antiguos |
| Ventana creciente | Ajusta con todo lo anterior | La usada aquí |
| Bloques con purga a ambos lados | Ajusta también con datos posteriores | No imita el uso real |

La ventana deslizante se vería así (10 años de ajuste). No se ha probado: es la mejora «ventana deslizante» del apartado 12.

In [9]:
deslizante = ex.fold_table(cfg, dev, horizon=5)
deslizante["ajuste_desde"] = deslizante["ajuste_hasta"] - pd.DateOffset(years=10)
plots.folds_timeline(deslizante, 5, manifest["ranges"]["test"])

<Figure size 1000x415 with 1 Axes>

**Brentcast** usa 4 cortes de 1 año (2018, 2019, 2020 y 2021), con el último 15 % de cada historia para parar y 2022 como margen antes de test. Aquí son 4 bloques de 2 años (2015–2023): más regímenes, pero la red de cada pliegue predice 2 años sin reentrenarse (decisión 5.5).

## 7. Resumen: lo que hay que saber decir

- La validación cruzada de series no baraja: avanza en el tiempo.
- La purga evita que un objetivo de entrenamiento contenga precios del bloque evaluado.
- Hay otras formas válidas (deslizante, más pliegues); no hay ninguna que permita barajar.
- Límite propio: a 252 filas cada bloque contiene unos 2 pronósticos independientes.

Siguiente cuaderno: `04_modelos.ipynb`.